# 논문 기반 Deep SVDD / Deep SAD 비교

정상 AE 사전학습 후 bias 없는 LeakyReLU 인코더와 고정 중심을 사용한다. Deep SAD는 정상 거리와 가스 역거리 손실(eta=1)을 이용한다. 저자 코드의 핵심 목적함수만 재구현한 소규모 센서 적용이며 논문 전체 실험 재현은 아니다. 비라벨 데이터는 이번 실험에서 추가하지 않는다.

https://proceedings.mlr.press/v80/ruff18a.html
https://arxiv.org/abs/1906.02694
https://github.com/lukasruff/Deep-SAD-PyTorch

두 시드 평균, 두 센서 관점 최대 점수 앙상블. 정상 기준 흔들림 4개 + 과거 잡음 대비 변화 8개. 10개 관측 구간 중첩 검증, 새 독립 검증 아님. CN7 가스는 하나의 이벤트 구간이라 일반화 근거가 약하다. 외부 평가 구간은 전처리·손실·중심·임계값·모델 선택에서 제외. 비율 .02/.05/.10은 내부 정상 점수 임계값 설정용이며 실제 오탐률 보장이 아니다.


In [1]:
from pathlib import Path
import json,copy,numpy as np,pandas as pd,torch
from torch import nn
from sklearn.metrics import confusion_matrix
torch.set_num_threads(1)
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/74_product_normalized_joint_gas_modes.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.rsplit('memo={}',1)[0])
memo={};logs=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 per_view=[]
 for v,(tr,te) in enumerate(views(ref,q)):
  tr=torch.tensor(np.clip(tr,-8,8),dtype=torch.float32)
  te=torch.tensor(np.clip(te,-8,8),dtype=torch.float32)
  normal=tr[y[ref]==0]
  seed_scores=[]
  for seed in [42,7]:
   torch.manual_seed(seed)
   enc=nn.Sequential(nn.Linear(tr.shape[1],16,bias=False),nn.LeakyReLU(.1),nn.Linear(16,4,bias=False))
   dec=nn.Sequential(nn.Linear(4,16),nn.LeakyReLU(.1),nn.Linear(16,tr.shape[1]))
   opt=torch.optim.Adam(list(enc.parameters())+list(dec.parameters()),lr=.001,weight_decay=1e-6)
   for epoch in range(40):
    opt.zero_grad();loss=((dec(enc(normal))-normal)**2).mean();loss.backward();opt.step()
   with torch.no_grad():
    c=enc(normal).mean(0);c=torch.where(c.abs()<.1,torch.where(c<0,-.1,.1),c)
   scores=[]
   for method in ['Deep_SVDD','Deep_SAD']:
    net=copy.deepcopy(enc)
    x=normal if method=='Deep_SVDD' else tr
    target=torch.tensor(y[ref]==1)
    opt=torch.optim.Adam(net.parameters(),lr=.001,weight_decay=1e-6)
    for epoch in range(80):
     opt.zero_grad();dist=((net(x)-c)**2).sum(1)
     loss=dist.mean() if method=='Deep_SVDD' else torch.where(target,1/(dist+1e-6),dist+1e-6).mean()
     assert torch.isfinite(loss)
     loss.backward();opt.step()
    with torch.no_grad():
     cal=((net(normal)-c)**2).sum(1).numpy()
     score=((net(te)-c)**2).sum(1).numpy()/max(float(np.quantile(cal,.95)),1e-6)
    scores.append(score)
    logs.append(dict(train=len(ref),query=len(q),view=v,seed=seed,method=method,normal_training=len(normal),gas_training=int(y[ref].sum()) if method=='Deep_SAD' else 0,loss=float(loss.detach()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
   seed_scores.append(np.column_stack(scores))
  per_view.append(np.mean(seed_scores,axis=0))
 memo[key]=np.max(per_view,axis=0)
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 outer=predict(ref,q);inner=np.full((len(ref),2),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['Deep_SVDD','Deep_SAD']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   ip=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],ip,labels=[0,1]).ravel()
   recall=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((recall,f1,-fp,name,budget,outer[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(outer[j,k]>cut),score=float(outer[j,k]),threshold=cut))
 selected=max(candidates,key=lambda a:a[:3]);choices.append(dict(block=int(b),selected=selected[3],budget=selected[4],inner_recall=selected[0]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(selected[5][j]),score=np.nan,threshold=np.nan))
 print('Finished Deep SVDD/SAD block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget,product),g in p.groupby(['model','budget','product']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,product=product,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn) if tp+fn else np.nan,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(logs),'training_audit')]:
 obj.to_csv(tables/f'gas_deep_svdd_sad_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


CN7 extra metadata 713 sensor-consistent unique inputs 713 conflicting 0


Finished Deep SVDD/SAD block 1


Finished Deep SVDD/SAD block 2


Finished Deep SVDD/SAD block 3


Finished Deep SVDD/SAD block 4


...

Finished Deep SVDD/SAD block 7


Finished Deep SVDD/SAD block 8


Finished Deep SVDD/SAD block 9


Finished Deep SVDD/SAD block 10


                      model  budget product  TP  FP  FN  TN   recall       F1
                   Deep_SAD    0.02     CN7   0   9  10 686 0.000000 0.000000
                   Deep_SAD    0.02     RG3   2  23  17 578 0.105263 0.090909
                   Deep_SAD    0.05     CN7   0  16  10 679 0.000000 0.000000
                   Deep_SAD    0.05     RG3   2  45  17 556 0.105263 0.060606
                   Deep_SAD    0.10     CN7   1  45   9 650 0.100000 0.035714
                   Deep_SAD    0.10     RG3   3  86  16 515 0.157895 0.055556
                  Deep_SVDD    0.02     CN7   0  19  10 676 0.000000 0.000000
                  Deep_SVDD    0.02     RG3   0   4  19 597 0.000000 0.000000
                  Deep_SVDD    0.05     CN7   1  34   9 661 0.100000 0.044444
                  Deep_SVDD    0.05     RG3   2  29  17 572 0.105263 0.080000
                  Deep_SVDD    0.10     CN7   3  81   7 614 0.300000 0.063830
                  Deep_SVDD    0.10     RG3   4  58  15 543 0.21